# 07. Analyzing Images Directly with Azure AI Content Safety

This notebook calls the **standalone Azure AI Content Safety** service directly — rather than
relying on Azure OpenAI/Foundry's automatic, embedded content filtering (notebooks 05-06) — to
analyze an image (`support.png`) and print per-category severity scores. This is the third and
final content-safety notebook in this section, and the one most directly aligned with AI-102's
dedicated **Content Safety** exam objectives.

**Difficulty:** Intermediate — introduces a new, separate Azure resource and SDK
(`azure-ai-contentsafety`) not used elsewhere in this section.

## Prerequisites

**Python packages** — `azure-ai-contentsafety` and `azure-core` are **not** currently in this
repo's root `requirements.txt`. Install them before running this notebook:
```bash
pip3 install azure-ai-contentsafety azure-core
```
> Note: the original script's comment said `pip3 install azure.ai.contentsafety` (dots) — the
> actual PyPI package name uses hyphens: `azure-ai-contentsafety`.

Also uses `python-dotenv` (already in `requirements.txt`).

**Azure resources**
- A standalone **Azure AI Content Safety** resource (created independently of any Azure OpenAI
  or Foundry project resource — Content Safety is its own Azure AI service).

**Environment variables** — add these to the root `.env`:
```
AZURE_CONTENTSAFETY_ENDPOINT=https://<your-content-safety-resource>.cognitiveservices.azure.com/
AZURE_CONTENTSAFETY_KEY=<your-content-safety-key>
```
These are new variable names, following the `AZURE_<SERVICE>_ENDPOINT` / `AZURE_<SERVICE>_KEY`
convention used for `AZURE_AI_PROJECT_ENDPOINT` etc. elsewhere in this repo.

**Input file:** `support.png` must exist in the same folder as this notebook (it already does).

## What You'll Learn

- How to call the **dedicated Content Safety API** directly via `ContentSafetyClient`, instead
  of relying on filtering embedded in an Azure OpenAI/Foundry call
- The `analyze_image()` method and its four built-in categories: **Hate, SelfHarm, Sexual,
  Violence**
- How severity is reported per category (`category_result.category`,
  `category_result.severity`)
- How this standalone, synchronous analysis approach compares to the automatic filtering seen in
  notebooks 05-06

### Imports and setup

- `ContentSafetyClient` — the dedicated client for the Content Safety service (a separate SDK
  package, `azure-ai-contentsafety`, from the `azure-ai-projects`/`openai` packages used
  elsewhere in this section).
- `AzureKeyCredential` — simple API-key authentication (from `azure-core`), as opposed to the
  `DefaultAzureCredential` (Entra ID) pattern used for the Foundry project client in earlier
  notebooks. Content Safety resources support key-based auth directly.
- `AnalyzeImageOptions`, `ImageData` — request model types specific to the image analysis call.
- The original script also imported `DefaultAzureCredential` and `AIProjectClient`, but never
  actually used them in the script body — likely leftover from copy-pasting the Foundry-based
  scripts (05/06). They're kept here for faithfulness to the original, but you can safely remove
  them since this notebook's logic doesn't touch a Foundry project at all.

💡 **Exam tip:** AI-102 treats **Azure AI Content Safety** as its own dedicated service with its
own resource type — distinct from the content filtering that's automatically bundled into Azure
OpenAI. Content Safety gives you direct, granular severity scores you can act on programmatically
(e.g. custom thresholds, human review queues), rather than just a pass/flag decision.

🔄 **Alternatives:** N/A for imports, but this whole notebook is the "explicit API call"
alternative to the automatic filtering shown in notebooks 05-06.

In [ ]:
import os

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

from azure.ai.contentsafety import ContentSafetyClient
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentsafety.models import AnalyzeImageOptions, ImageData
from dotenv import load_dotenv

load_dotenv()

### Building the Content Safety client

- `CONTENT_SAFETY_ENDPOINT` and `CONTENT_SAFETY_KEY` are read from the environment instead of
  being hardcoded (the original script left the key as an empty string placeholder).
- `ContentSafetyClient(endpoint, credential)` — constructed with `AzureKeyCredential`, Content
  Safety's standard key-based authentication.

🔄 **Alternatives:** `ContentSafetyClient` also supports `DefaultAzureCredential`-based Entra ID
auth (swap `AzureKeyCredential(key)` for a `DefaultAzureCredential()` instance) if you'd rather
avoid managing a key, consistent with the auth pattern used for the Foundry project client in
earlier notebooks.

In [ ]:
CONTENT_SAFETY_ENDPOINT = os.getenv("AZURE_CONTENTSAFETY_ENDPOINT")
CONTENT_SAFETY_KEY = os.getenv("AZURE_CONTENTSAFETY_KEY")

content_safety_client = ContentSafetyClient(
    endpoint=CONTENT_SAFETY_ENDPOINT,
    credential=AzureKeyCredential(CONTENT_SAFETY_KEY)
)

### Preparing the image analysis request

- Reads `support.png` as raw bytes (no base64 encoding needed here — `ImageData` accepts raw
  bytes directly via its `content` field, unlike the `data:` URI approach used with the OpenAI
  Responses API in notebooks 01/06).
- `ImageData(content=image_bytes)` wraps those bytes in the SDK's request model.
- `AnalyzeImageOptions(image=...)` wraps that into the final options object passed to
  `analyze_image()`.

💡 **Exam tip:** Content Safety's `analyze_image` call is **synchronous** — it returns the
severity results directly in the HTTP response, unlike some other Azure AI services that use an
async submit-then-poll pattern. Know that both `analyze_text` and `analyze_image` exist as
sibling synchronous methods for their respective content types.

In [ ]:
IMAGE_PATH = "support.png"

with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

image_request = AnalyzeImageOptions(
    image=ImageData(content=image_bytes)
)

### Running the analysis and reading results

- `content_safety_client.analyze_image(image_request)` sends the request and returns a result
  object synchronously.
- `moderation_result.categories_analysis` is a list of per-category results — one entry per
  built-in category (Hate, SelfHarm, Sexual, Violence) — each with a `.category` name and a
  `.severity` score.

💡 **Exam tip:** Content Safety's four built-in categories are **Hate, SelfHarm, Sexual, and
Violence**. Severity is reported on a numeric scale (higher = more severe); Content Safety also
supports **custom blocklists** for exact-match text you want flagged regardless of category —
but blocklists are a text-only feature, not applicable to image analysis like this call.

🔄 **Alternatives:** For text instead of images, the sibling call is
`content_safety_client.analyze_text(AnalyzeTextOptions(text=...))`, returning the same shape of
per-category severity results — plus, for text specifically, the option to layer custom
blocklists on top of the built-in categories.

In [ ]:
moderation_result = content_safety_client.analyze_image(image_request)

print("Image moderation result:")
for category_result in moderation_result.categories_analysis:
    print(category_result.category, category_result.severity)

## Summary

This notebook called Azure AI Content Safety's `analyze_image()` directly against `support.png`,
using a dedicated Content Safety resource and API key rather than relying on filtering embedded
in an Azure OpenAI/Foundry call. The result is a clean list of per-category severity scores
(Hate, SelfHarm, Sexual, Violence), giving direct, structured signal you can act on
programmatically — e.g. routing anything above a chosen severity threshold to human review. This
closes out the section's progression: generate (02) → prompt-edit (03) → mask-edit (04) →
automatic text filtering (05) → automatic image filtering (06) → explicit, standalone Content
Safety analysis (07).

## Try It Yourself

1. **Easy:** Run this notebook against a different image in this folder (e.g.
   `generated_image.png` or `discover.png`) and compare severity scores across images.
2. **Intermediate:** Adapt the code to call `analyze_text()` instead of `analyze_image()` on the
   same harmful prompt string used in notebook 05, and compare the category/severity output
   shape between the two content types.
3. **Advanced:** Look up Content Safety's custom **blocklist** feature (text-only) in the Azure
   docs, create a blocklist with a term relevant to your own use case, and extend an
   `analyze_text()` call to check matches against it alongside the built-in categories.